In [6]:
import math
import os
import random

import numpy as np
from torch.utils.data import Dataset, DataLoader
import nibabel
from scipy import ndimage
import time
import torch
import torch.nn as nn

import SimpleITK as sitk

In [7]:
class NeuroVascularSegmentDS(Dataset):
    def __init__(self, root_dir, img_list, phase,crop_size, scale_size):
        self.series_list = []
        self.images_list = []
        self.masks_list = []
        self.phase = phase
        self.root_dir = root_dir
        self.crop_size = crop_size
        self.scale_size = scale_size
        with open(img_list, 'r') as f:
            for line in f.readlines():
                line = line.strip()
                if line is None or len(line) == 0:
                    continue
                ss = line.split('\t')
                if len(ss) != 2:
                    continue
                image_file = os.path.join(root_dir, ss[0])
                if not os.path.isfile(image_file):
                    continue
                mask_file = os.path.join(root_dir, ss[1])
                if not os.path.isfile(mask_file):
                    continue
                self.images_list.append(image_file)
                self.masks_list.append(mask_file)
                
    def __random_crop_data(self, volume, mask, size):
        [img_d, img_h, img_w] = volume.shape
        [input_d, input_h, input_w] = size
        z_min_upper = img_d - input_d
        y_min_upper = img_h - input_h
        x_min_upper = img_w - input_w
        Z_min = np.random.randint(0, z_min_upper)
        Y_min = np.random.randint(0, y_min_upper)
        X_min = np.random.randint(0, x_min_upper)

        Z_max = Z_min + input_d
        Y_max = Y_min + input_h
        X_max = X_min + input_w
#         print('x:[{}-{}]\ty:[{}-{}]\tz:[{}-{}]'.format(X_min, X_max, Y_min, Y_max, Z_min, Z_max))
#         print(mask.dtype)
#         print(mask.max())
        return volume[Z_min: Z_max, Y_min: Y_max, X_min: X_max], mask[Z_min: Z_max, Y_min: Y_max, X_min: X_max]
    
    def __len__(self):
        return len(self.images_list)
    
    def __getitem__(self, idx):
        if self.phase == 'train':
            volume_path = self.images_list[idx]
            mask_path = self.masks_list[idx]
            print(mask_path)
            with open(volume_path, 'rb') as f:
                    volume_data = np.load(f)
            with open(mask_path, 'rb') as f:
                    mask_data = np.load(f)
            print(volume_data.shape)
            cropped_volume, cropped_mask = self.__random_crop_data(volume_data, mask_data, self.crop_size)
            cropped_volume = torch.from_numpy(cropped_volume).float()
            cropped_volume = torch.unsqueeze(cropped_volume, axis=0)
            return cropped_volume, cropped_mask

In [3]:
crop_size = [56, 224, 224]
root_dir = '../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed'
config_file = '../data/brain_henan/algo_mask/config/neuro_vascular_seg_mix_0_1.txt'
ds = NeuroVascularSegmentDS(root_dir, config_file, 'train', crop_size, crop_size)

dataloader = DataLoader(ds, batch_size=1, num_workers=8, shuffle=True, pin_memory=False)
cnt_total = 0
cnt_valid = 0
for index, (images, masks) in enumerate(dataloader):
    cnt_total += 1
    max_val = masks.max()
    if max_val == 1:
        cnt_valid += 1

../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000016010904522334300001290_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000017051013402768700000128_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000019012215113151500007098_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000018113015131734300000214_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000019033013292326500020192_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.99.2.9594.30000019042616245185900000128_mask.npy
../data/brain_henan/algo_mask/brain_seg_by_threshold_preprocessed/1.3.12.2.1107.5.9

In [5]:
print(cnt_valid/cnt_total)

0.9545454545454546
